# Day 10 — LangGraph Wrap-up + n8n Workflow Automation

We finish LangGraph by turning an agent from a **black box** into a **glass box**:
persist its state to disk, inspect the full trace of what it did, and even *time-travel*
back to any earlier step. Then we map the same **trigger → nodes → response** shape onto
**n8n**, the low-code canvas that lets non-engineers build AI workflows without Python.

## Objectives
- Persist graph state to a **file** with `SqliteSaver` so an agent survives a full restart.
- Read the **trace** with `get_state()` / `get_state_history()` — debug, don't guess.
- **Time-travel**: resume a run from any past checkpoint using its `config`.
- Map LangGraph (code) onto **n8n** (low-code) and reproduce n8n's first workflow in Python.

> **Backend:** [Groq](https://console.groq.com) · `llama-3.3-70b-versatile` (only the optional
> n8n-style cell needs a key). The core checkpointing demo runs with **zero setup**.

In [ ]:
# Day 10 adds the SQLite checkpointer on top of the usual LangGraph stack
!pip install -q langgraph langchain langchain-groq langgraph-checkpoint-sqlite python-dotenv
print("✅ Libraries installed.")

## 💾 1 · From in-memory to on-disk checkpointing

On Day 8 we used `InMemorySaver` — great, but it **vanishes when the program stops**.
To make an agent truly resumable across restarts, save checkpoints to a **file** with
`SqliteSaver`. Same graph, one line different.

The graph below uses tiny **deterministic** nodes (`plan → research → answer`) so the whole
persistence demo runs without an API key. In a real agent these nodes would call the LLM
exactly like Day 9 — checkpointing behaves identically either way.

In [ ]:
import os, sqlite3
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

# --- Shared state passed between nodes -------------------------------------
class TaskState(TypedDict):
    question: str
    plan: list
    research: str
    answer: str

def plan_node(state: TaskState):
    print("🧭 plan_node")
    return {"plan": ["gather facts", "draft answer"]}

def research_node(state: TaskState):
    print("🔎 research_node")
    return {"research": f"facts about: {state['question']}"}

def answer_node(state: TaskState):
    print("✍️ answer_node")
    return {"answer": f"Based on {state['research']}, here is the answer."}

# --- Wire the graph --------------------------------------------------------
builder = StateGraph(TaskState)
builder.add_node("plan", plan_node)
builder.add_node("research", research_node)
builder.add_node("answer", answer_node)
builder.add_edge(START, "plan")
builder.add_edge("plan", "research")
builder.add_edge("research", "answer")
builder.add_edge("answer", END)

# 🗄️ persist to a real file on disk — survives restarts
conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
graph = builder.compile(checkpointer=SqliteSaver(conn))   # 👈 the only change vs in-memory
print("✅ Graph compiled with an on-disk SqliteSaver checkpointer.")

## ⏯️ 2 · Run, stop, restart, resume

Because state now lives in a file, you can **close the whole program** and still pick up
exactly where you left off — just reconnect the **same** file and reuse the same `thread_id`.

In [ ]:
config = {"configurable": {"thread_id": "session-1"}}

# ▶️ first run — every super-step is auto-saved into checkpoints.sqlite
graph.invoke(
    {"question": "Draft a plan for a science fair", "plan": [], "research": "", "answer": ""},
    config,
)

# ...imagine the entire program stops here, then restarts...
# reconnect the SAME file and rebuild the graph — nothing is carried over in RAM
conn2 = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
graph2 = builder.compile(checkpointer=SqliteSaver(conn2))

snapshot = graph2.get_state(config)   # 🔎 state is still there, loaded from disk
print("\n--- Resumed after a simulated restart ---")
print("answer:", snapshot.values.get("answer"))
print("next  :", snapshot.next, "(empty = the run finished)")

## 🐾 3 · Inspecting what happened (the trace)

A black-box agent is a nightmare to debug. Checkpointing gives you a **trace**: every step
it took, in order. `get_state_history()` walks the full timeline, newest first — so you can
pinpoint exactly which step changed which value.

In [ ]:
print("--- State history (trace) ---")
for snap in graph2.get_state_history(config):
    step = snap.metadata.get("step")
    print(f"step {step:>2} · next={snap.next} · keys written={list(snap.values.keys())}")

## ⏪ 4 · Time-travel debugging

Each snapshot carries a `config` you can pass back to **resume from that exact point** —
rerun from step 1 instead of the start. That's "time travel," and it falls out of
checkpointing for free.

In [ ]:
history = list(graph2.get_state_history(config))
print(f"{len(history)} checkpoints available.")

# pick an earlier checkpoint (index 2 = a mid-run step) and replay forward from it
earlier = history[min(2, len(history) - 1)]
print("⏪ Replaying from step:", earlier.metadata.get("step"))
graph2.invoke(None, earlier.config)   # None resumes forward from that exact checkpoint
print("✅ Re-ran forward from a past checkpoint — debugging by replay, not guesswork.")

---
# Extended Implementation — code vs low-code (n8n)

n8n's **trigger → nodes → response** is the *same idea* as LangGraph's **START → nodes → END**,
just drawn on a canvas instead of written in code:

| LangGraph (code) | n8n (low-code) |
|------------------|----------------|
| `START` | ⚡ Trigger node |
| a node function | 🟦 a node on the canvas |
| `state["x"]` | 🔗 `{{ $json.x }}` expression |
| `add_edge(a, b)` | drawing a line a → b |
| conditional edge (Day 8) | 🔀 Switch node |
| human-in-the-loop (Day 9) | a human-review branch |

n8n itself has no Python, but its first tutorial workflow — **⚡ Webhook → 🤖 LLM → ↩️ Respond** —
is easy to reproduce in code so you can see the equivalence. The cell below mirrors it:
it reads the trigger payload, fills an expression, calls the LLM (real if `GROQ_API_KEY`
is set, otherwise a stub), and returns the response.

In [ ]:
def n8n_style_workflow(payload: dict) -> dict:
    """The n8n 'first workflow' expressed in Python: Webhook -> LLM -> Respond."""
    question = payload.get("body", {}).get("question", "")     # ⚡ trigger data
    prompt = f"Answer this clearly: {question}"                # 🔗 expression {{ $json.body.question }}
    api_key = os.getenv("GROQ_API_KEY")
    if api_key:
        from langchain_groq import ChatGroq
        llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=api_key, temperature=0)
        text = llm.invoke(prompt).content                      # 🤖 LLM node
    else:
        text = f"[simulated answer — set GROQ_API_KEY for a real reply] {question}"
    return {"text": text}                                      # ↩️ respond node

result = n8n_style_workflow({"body": {"question": "What is n8n in one sentence?"}})
print(result["text"])

## 🎓 Recap & cheat sheet

- 💾 **`SqliteSaver`** — persist graph state to a disk file that survives restarts.
- 🔎 **`get_state()` / `get_state_history()`** — inspect the full trace; debug, don't guess.
- ⏪ **Time travel** — resume from any past checkpoint using its `config`.
- 🧰 **n8n** — visual, low-code AI workflows: **trigger → nodes → response**.
- ⚖️ **Choose deliberately** — low-code for standard automations owned by non-engineers;
  code for control, testing, and complex multi-agent orchestration.

```python
# LANGGRAPH: PERSIST TO DISK
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
graph = builder.compile(checkpointer=SqliteSaver(conn))
config = {"configurable": {"thread_id": "session-1"}}
graph.invoke(inputs, config)             # runs + auto-saves
graph.get_state(config)                   # current state
list(graph.get_state_history(config))     # full trace
```

```bash
# N8N: START IT  (needs Node.js 18+)
npx n8n                  # -> open http://localhost:5678
npx n8n start --tunnel   # -> public URL for external webhooks
```

> **Day 10 outcome:** a complete, debuggable LangGraph agent on one side and a low-code
> AI workflow on the other — the engineer *and* non-engineer delivery paths.